# ECAPA-TDNN baseline — standard recipe (VoxCeleb2 → VoxCeleb1)

A conventional reference point for the triplet-STDP fingerprint pipeline in this directory.
Trained the **standard way** — the reference recipe from
[TaoRuijie/ECAPA-TDNN](https://github.com/TaoRuijie/ECAPA-TDNN) (EER 0.86 on Vox1-O when trained
on VoxCeleb2 only), not a capacity- or schedule-matched variant of `ann_backend.ipynb`.

| | |
|---|---|
| model | ECAPA-TDNN, `C=1024`, 192-d embedding (~15.4M params) |
| features | 80-mel, 16 kHz, n_fft 512 / win 400 / hop 160, 20–7600 Hz, pre-emphasis 0.97 |
| train input | 2 s random crop |
| loss | AAMSoftmax, m=0.2, s=30 |
| optim | Adam, lr 1e-3, wd 2e-5, lr × 0.97 per epoch, 80 epochs |
| batch | 128 — the one deviation from the reference, which uses 400; sized for a 16 GB Kaggle GPU at `C=1024` |
| augment | SpecAugment (`FbankAug`) always; MUSAN + RIR if those datasets are attached |
| train data | full VoxCeleb2 dev, **1 random utterance per session, all sessions** |
| eval data | full VoxCeleb1 dev, same 1-per-session rule |

**Eval is the same protocol as `ann_backend.ipynb`** — session-free all-pairs cosine retrieval
giving EER / Rank-1 / Rank-5 / mAP — *not* the Vox1-O trial list. The point is a number directly
comparable to the SNN runs.

### Why there is a decode-cache cell

The SNN consumed each wav once and emitted a small fingerprint. ECAPA re-reads audio every epoch,
and decoding ~145k m4a files per epoch would make training CPU-bound. Cell 4 decodes once into a
**ragged int16 memmap on scratch disk** (~28 GB train at a 6 s cap, ~6 GB eval at 15 s) with an
`(offset, length)` index. Reads come back through the page cache, so RSS stays flat. This costs
~25–30 min once per session; training itself resumes from checkpoint.

In [ ]:
# ── Environment check (Kaggle ships torch, torchaudio, soundfile, scipy, ffmpeg) ──
import shutil, sys
import torch, torchaudio
print("torch     ", torch.__version__)
print("torchaudio", torchaudio.__version__)
print("cuda      ", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert shutil.which("ffmpeg"), "ffmpeg not on PATH — needed to decode VoxCeleb2 .m4a"
print("ffmpeg    ", shutil.which("ffmpeg"))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG  —  the only cell you normally edit
# ══════════════════════════════════════════════════════════════════════════════
import os, glob, math, time, random
import numpy as np
import torch

SEED = 1234
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---- Data roots -------------------------------------------------------------
# Both layouts are handled: person/session/wav is read off the tail of each path,
# so vox1's extra dev_NN level needs no special case.
TRAIN_GLOB = "/kaggle/input/datasets/qphulong/vox2-voices-200person-shard*"
TEST_GLOB  = "/kaggle/input/datasets/qphulong/vox1-voices"
AUDIO_EXTS = (".m4a", ".wav")   # both corpora are m4a; .wav kept as a harmless fallback

# ---- Optional MUSAN / RIR (standard recipe uses them; skipped if absent) -----
MUSAN_GLOB = "/kaggle/input/**/musan"
RIR_GLOB   = "/kaggle/input/**/RIRS_NOISES/simulated_rirs"
MAX_RIR    = 4000          # cap how many RIR impulse responses to hold in RAM

# ---- Audio ------------------------------------------------------------------
SR             = 16000
CROP_SEC       = 2.0                       # train crop (standard ECAPA)
CROP_SAMPLES   = int(CROP_SEC * SR)        # 32000 -> 201 frames
TRAIN_CAP_SEC  = 6.0                       # stored per train clip (crops drawn from this)
TEST_CAP_SEC   = 15.0                      # stored per eval clip (full-utterance eval)

# ---- Decode cache (scratch disk, NOT /kaggle/working which is output-quota'd) --
CACHE_DIR = "/kaggle/temp/ecapa_cache"

# ---- Model ------------------------------------------------------------------
C          = 1024        # reference default; 512 (~7.0M params) also standard
EMBED_DIM  = 192

# ---- Optim / schedule (reference recipe) ------------------------------------
EPOCHS       = 80
BATCH_SIZE   = 128       # reference uses 400; 128 is sized for a 16GB Kaggle GPU at C=1024
LR           = 1e-3
LR_DECAY     = 0.97      # multiplied per epoch
WEIGHT_DECAY = 2e-5
AAM_M        = 0.2
AAM_S        = 30
GRAD_CLIP    = 5.0

# ---- Eval -------------------------------------------------------------------
EVAL_EVERY   = 1
EVAL_BATCH   = 32        # length-sorted buckets

# ---- Checkpointing ----------------------------------------------------------
RESUME    = True
CKPT_DIR  = "/kaggle/working"
TAG       = f"ecapa_C{C}"
LAST_CKPT = os.path.join(CKPT_DIR, f"last_{TAG}.pt")
BEST_CKPT = os.path.join(CKPT_DIR, f"best_{TAG}.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"device={device}  TAG={TAG}  C={C}  batch={BATCH_SIZE}  epochs={EPOCHS}")
print(f"cache  ={CACHE_DIR}")

In [ ]:
# ── Enumerate: 1 random utterance per session, every session, every speaker ───
# person/session are the last two directories before the file, which is true for
# both  {shard}/{person}/{session}/x.m4a  and  {root}/dev_NN/{person}/{session}/x.wav
from pathlib import Path

def enumerate_corpus(pattern, name, seed=SEED):
    roots = sorted(glob.glob(pattern))
    if not roots:
        try:
            avail = sorted(os.listdir(os.path.dirname(pattern.rstrip("*")) or "/"))[:20]
        except OSError:
            avail = "<parent directory does not exist>"
        raise FileNotFoundError(f"[{name}] nothing matches {pattern!r}. "
                                f"Siblings of that path: {avail}")
    sessions = {}                                    # (person, session) -> [paths]
    for root in roots:
        for dirpath, _, filenames in os.walk(root):
            hits = [f for f in filenames if f.lower().endswith(AUDIO_EXTS)]
            if not hits:
                continue
            parts = Path(dirpath).parts
            person, session = parts[-2], parts[-1]
            sessions.setdefault((person, session), []).extend(
                os.path.join(dirpath, f) for f in sorted(hits))

    rng = np.random.default_rng(seed)
    entries = []
    for (person, session) in sorted(sessions):
        wavs = sorted(sessions[(person, session)])
        entries.append(dict(person_id=person, record_id=session,
                            wav_path=wavs[rng.integers(len(wavs))]))

    n_spk = len({e["person_id"] for e in entries})
    print(f"[{name}] {len(roots)} root(s) | {n_spk} speakers | "
          f"{len(entries)} sessions -> {len(entries)} utterances (1 per session)")
    return entries

train_entries = enumerate_corpus(TRAIN_GLOB, "train/vox2")
test_entries  = enumerate_corpus(TEST_GLOB,  "eval/vox1")

_dev_spk  = {e["person_id"] for e in train_entries}
_test_spk = {e["person_id"] for e in test_entries}
_overlap  = _dev_spk & _test_spk
assert not _overlap, f"train/eval speaker overlap: {sorted(_overlap)[:10]}"
print(f"[disjoint] train={len(_dev_spk)}  eval={len(_test_spk)}  overlap=0")

In [ ]:
# ── Decode once -> ragged int16 memmap on scratch disk ───────────────────────
# A *thread* pool, deliberately, not a process pool: the ffmpeg path spends its
# time in a subprocess and soundfile releases the GIL, so threads parallelize the
# real work — while avoiding both fork-after-torch-init (which can abort the
# kernel) and the pickling of notebook-defined functions that a process pool needs.
# Results are written sequentially by the caller so offsets stay aligned with
# `entries` order. Ragged packing (not a padded 2-D array) — padding every clip to
# the cap would waste several GB.
from concurrent.futures import ThreadPoolExecutor
import shutil, subprocess
import soundfile as sf

def _decode_ffmpeg(path, sr):
    proc = subprocess.run(
        ["ffmpeg", "-nostdin", "-loglevel", "error", "-i", path,
         "-ac", "1", "-ar", str(sr), "-f", "f32le", "-acodec", "pcm_f32le", "-"],
        capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.decode("utf-8", "ignore")[:200])
    return np.frombuffer(proc.stdout, dtype="<f4").astype(np.float32)

_FFMPEG_EXTS = (".m4a", ".aac", ".mp4", ".m4b")

def load_audio(path, sr=SR):
    """Mono float32 at `sr`.

    Both VoxCeleb1 and VoxCeleb2 are m4a here, so ffmpeg is the normal path and is
    dispatched on extension — a try/except around soundfile would raise once per
    file for all ~167k of them. soundfile still serves any wav/flac that appear."""
    if path.lower().endswith(_FFMPEG_EXTS):
        return _decode_ffmpeg(path, sr)
    try:
        y, sr_native = sf.read(path, dtype="float32", always_2d=False)
    except Exception:
        return _decode_ffmpeg(path, sr)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr_native != sr:
        import librosa
        y = librosa.resample(y, orig_sr=sr_native, target_sr=sr)
    return np.ascontiguousarray(y, dtype=np.float32)

def _decode_one(args):
    path, cap = args
    try:
        y = load_audio(path, SR)
    except Exception:
        return None
    if y.size == 0:
        return None
    if y.size > cap:
        y = y[:cap]
    return np.clip(y * 32768.0, -32768, 32767).astype(np.int16)

def build_cache(entries, name, cap_sec, workers=None):
    """Decode `entries` into {CACHE_DIR}/{name}.bin + {name}_index.npz. Idempotent."""
    cap = int(cap_sec * SR)
    bin_path = os.path.join(CACHE_DIR, f"{name}.bin")
    idx_path = os.path.join(CACHE_DIR, f"{name}_index.npz")

    if os.path.exists(bin_path) and os.path.exists(idx_path):
        z = np.load(idx_path, allow_pickle=True)
        if int(z["n_entries"]) == len(entries) and int(z["cap"]) == cap \
           and os.path.getsize(bin_path) == int(z["nbytes"]):
            print(f"[{name}] cache hit: {int(z['n_kept'])} clips, "
                  f"{int(z['nbytes'])/1e9:.2f} GB — skipping decode")
            return idx_path
        print(f"[{name}] cache stale (entries/cap/size mismatch) — rebuilding")

    # Pre-flight: most VoxCeleb utterances are longer than the cap, so the cache
    # lands near the worst case. Fail now rather than 20 minutes into the decode.
    est  = len(entries) * cap * 2 * 0.92
    free = shutil.disk_usage(CACHE_DIR).free
    print(f"[{name}] ~{est/1e9:.1f} GB estimated, {free/1e9:.1f} GB free on {CACHE_DIR}")
    if est > free:
        raise RuntimeError(
            f"[{name}] not enough scratch space: need ~{est/1e9:.1f} GB, have "
            f"{free/1e9:.1f} GB. Lower TRAIN_CAP_SEC / TEST_CAP_SEC (train only ever "
            f"draws {CROP_SEC}s crops, so 4.0 is a safe reduction) or point CACHE_DIR "
            f"at a bigger volume.")

    # Every file is m4a -> every decode is an ffmpeg subprocess, so the threads are
    # mostly waiting on process spawn rather than burning CPU. Oversubscribe.
    workers = workers or (os.cpu_count() or 2) * 3
    paths = [e["wav_path"] for e in entries]
    offsets = np.zeros(len(paths), dtype=np.int64)
    lengths = np.zeros(len(paths), dtype=np.int64)

    t0, pos, n_fail, i = time.time(), 0, 0, 0
    CHUNK = 512    # bounds in-flight decoded audio to ~CHUNK * cap * 2 bytes (~100 MB)
    with open(bin_path, "wb", buffering=1 << 22) as f, \
         ThreadPoolExecutor(max_workers=workers) as pool:
        for s in range(0, len(paths), CHUNK):
            batch = paths[s:s + CHUNK]
            for arr in pool.map(_decode_one, [(p, cap) for p in batch]):
                if arr is None:
                    n_fail += 1
                    offsets[i], lengths[i] = pos, 0
                else:
                    f.write(arr.tobytes())
                    offsets[i], lengths[i] = pos, arr.size
                    pos += arr.size
                i += 1
            if s % (CHUNK * 20) == 0 and s:
                el = time.time() - t0
                print(f"  [{name}] {i}/{len(paths)}  {pos*2/1e9:.2f} GB  "
                      f"{el:.0f}s  eta {el/i*(len(paths)-i):.0f}s", flush=True)

    keep = np.nonzero(lengths)[0]
    np.savez(idx_path,
             offsets=offsets[keep], lengths=lengths[keep],
             person_ids=np.array([entries[i]["person_id"] for i in keep]),
             record_ids=np.array([entries[i]["record_id"] for i in keep]),
             n_entries=len(entries), n_kept=len(keep), cap=cap, nbytes=pos * 2)
    print(f"[{name}] decoded {len(keep)}/{len(paths)} clips ({n_fail} failed) "
          f"-> {pos*2/1e9:.2f} GB in {time.time()-t0:.0f}s")
    return idx_path

train_idx = build_cache(train_entries, "train", TRAIN_CAP_SEC)
test_idx  = build_cache(test_entries,  "test",  TEST_CAP_SEC)

In [ ]:
# ── Waveform bank + optional MUSAN/RIR augmentation ──────────────────────────
# No DataLoader: reads are memmap slices out of the page cache, so a plain bank
# with .batch(idx) is both faster and free of the worker-caching leak that bit
# the fingerprint backend.
import scipy.signal

def encode_labels(pid):
    classes = sorted(set(pid.tolist()))
    c2i = {c: i for i, c in enumerate(classes)}
    return np.fromiter((c2i[p] for p in pid), dtype=np.int64, count=len(pid)), len(classes)

class WaveAugment:
    """MUSAN additive noise + RIR reverb, voxceleb_trainer settings.
       Inactive (returns input unchanged) when the datasets are not attached."""
    SNR = {"noise": (0, 15), "speech": (13, 20), "music": (5, 15)}
    NUM = {"noise": (1, 1),  "speech": (3, 7),   "music": (1, 1)}

    def __init__(self, musan_glob, rir_glob, max_rir=MAX_RIR):
        self.noise = {}
        self.rirs  = []
        musan_roots = glob.glob(musan_glob, recursive=True)
        for cat in ("noise", "speech", "music"):
            files = []
            for r in musan_roots:
                files += glob.glob(os.path.join(r, cat, "**", "*.wav"), recursive=True)
            if files:
                self.noise[cat] = files
        for r in glob.glob(rir_glob, recursive=True):
            self.rirs += glob.glob(os.path.join(r, "**", "*.wav"), recursive=True)
        if len(self.rirs) > max_rir:
            self.rirs = random.Random(SEED).sample(self.rirs, max_rir)
        self.active = bool(self.noise) or bool(self.rirs)
        if self.active:
            print(f"[augment] MUSAN "
                  f"{ {k: len(v) for k, v in self.noise.items()} } | RIR {len(self.rirs)}")
        else:
            print("[augment] MUSAN/RIR not found — SpecAugment only "
                  "(set MUSAN_GLOB / RIR_GLOB to enable the full standard recipe)")

    def _noise_clip(self, path, n):
        info = sf.info(path)
        if info.frames > n:
            start = random.randint(0, info.frames - n - 1)
            y, _ = sf.read(path, start=start, frames=n, dtype="float32", always_2d=False)
        else:
            y, _ = sf.read(path, dtype="float32", always_2d=False)
            y = np.pad(y, (0, n - y.size), mode="wrap")
        if y.ndim > 1:
            y = y.mean(axis=1)
        return y.astype(np.float32)

    def _add_noise(self, wav, cat):
        n = wav.size
        clean_db = 10 * np.log10(np.mean(wav ** 2) + 1e-4)
        lo, hi = self.NUM[cat]
        acc = np.zeros(n, dtype=np.float32)
        for path in random.sample(self.noise[cat], min(random.randint(lo, hi),
                                                       len(self.noise[cat]))):
            try:
                y = self._noise_clip(path, n)
            except Exception:
                continue
            n_db = 10 * np.log10(np.mean(y ** 2) + 1e-4)
            snr  = random.uniform(*self.SNR[cat])
            acc += np.sqrt(10 ** ((clean_db - n_db - snr) / 10)) * y
        return wav + acc

    def _reverb(self, wav):
        try:
            rir, _ = sf.read(random.choice(self.rirs), dtype="float32", always_2d=False)
        except Exception:
            return wav
        if rir.ndim > 1:
            rir = rir.mean(axis=1)
        rir = rir / (np.sqrt(np.sum(rir ** 2)) + 1e-8)
        return scipy.signal.fftconvolve(wav, rir, mode="full")[:wav.size].astype(np.float32)

    def __call__(self, wav):
        if not self.active:
            return wav
        choices = ([0] + ([1] if self.rirs else [])
                       + [c for c in ("speech", "music", "noise") if c in self.noise])
        pick = random.choice(choices)
        if pick == 0:
            return wav
        if pick == 1:
            return self._reverb(wav)
        return self._add_noise(wav, pick)

class WaveBank:
    """Ragged int16 memmap -> float32 crops. `augment=None` disables waveform aug."""
    def __init__(self, bin_path, idx_path, augment=None):
        z = np.load(idx_path, allow_pickle=True)
        self.offsets = z["offsets"]; self.lengths = z["lengths"]
        self.person_ids = z["person_ids"].astype(str)
        self.record_ids = z["record_ids"].astype(str)
        self.wav = np.memmap(bin_path, dtype=np.int16, mode="r")
        self.augment = augment
        self.n = len(self.offsets)

    def raw(self, i, cap=None):
        o, L = int(self.offsets[i]), int(self.lengths[i])
        if cap is not None:
            L = min(L, cap)
        return np.asarray(self.wav[o:o + L], dtype=np.float32) / 32768.0

    def crop(self, i, n):
        """Random n-sample crop; wrap-pad clips shorter than n (voxceleb_trainer)."""
        o, L = int(self.offsets[i]), int(self.lengths[i])
        if L < n:
            y = np.asarray(self.wav[o:o + L], dtype=np.float32) / 32768.0
            return np.pad(y, (0, n - L), mode="wrap")
        s = random.randint(0, L - n)
        return np.asarray(self.wav[o + s:o + s + n], dtype=np.float32) / 32768.0

    def batch(self, idx, n=CROP_SAMPLES):
        out = np.empty((len(idx), n), dtype=np.float32)
        for k, i in enumerate(idx):
            w = self.crop(int(i), n)
            out[k] = self.augment(w) if self.augment is not None else w
        return torch.from_numpy(out)

wave_aug   = WaveAugment(MUSAN_GLOB, RIR_GLOB)
train_bank = WaveBank(os.path.join(CACHE_DIR, "train.bin"), train_idx, augment=wave_aug)
test_bank  = WaveBank(os.path.join(CACHE_DIR, "test.bin"),  test_idx,  augment=None)

y_train, NUM_CLASSES = encode_labels(train_bank.person_ids)
y_test,  _           = encode_labels(test_bank.person_ids)
print(f"[bank] train {train_bank.n} clips / {NUM_CLASSES} speakers | "
      f"eval {test_bank.n} clips / {len(set(y_test.tolist()))} speakers")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  ECAPA-TDNN — ported verbatim from https://github.com/TaoRuijie/ECAPA-TDNN
#  (itself derived from clovaai/voxceleb_trainer, lawlict/ECAPA-TDNN, speechbrain)
#  Only deviation: the fbank front-end is forced to fp32 so it is safe under AMP.
# ═════════════════════════════════════════════════════════════════════════════
import math
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F

class SEModule(nn.Module):
    def __init__(self, channels, bottleneck=128):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, bottleneck, kernel_size=1, padding=0),
            nn.ReLU(),
            nn.Conv1d(bottleneck, channels, kernel_size=1, padding=0),
            nn.Sigmoid())

    def forward(self, x):
        return x * self.se(x)

class Bottle2neck(nn.Module):
    def __init__(self, inplanes, planes, kernel_size=None, dilation=None, scale=8):
        super().__init__()
        width = int(math.floor(planes / scale))
        self.conv1 = nn.Conv1d(inplanes, width * scale, kernel_size=1)
        self.bn1   = nn.BatchNorm1d(width * scale)
        self.nums  = scale - 1
        num_pad = math.floor(kernel_size / 2) * dilation
        self.convs = nn.ModuleList([
            nn.Conv1d(width, width, kernel_size=kernel_size, dilation=dilation,
                      padding=num_pad) for _ in range(self.nums)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(width) for _ in range(self.nums)])
        self.conv3 = nn.Conv1d(width * scale, planes, kernel_size=1)
        self.bn3   = nn.BatchNorm1d(planes)
        self.relu  = nn.ReLU()
        self.width = width
        self.se    = SEModule(planes)

    def forward(self, x):
        residual = x
        out = self.bn1(self.relu(self.conv1(x)))
        spx = torch.split(out, self.width, 1)
        for i in range(self.nums):
            sp = spx[i] if i == 0 else sp + spx[i]
            sp = self.bns[i](self.relu(self.convs[i](sp)))
            out = sp if i == 0 else torch.cat((out, sp), 1)
        out = torch.cat((out, spx[self.nums]), 1)
        out = self.bn3(self.relu(self.conv3(out)))
        return self.se(out) + residual

class PreEmphasis(nn.Module):
    def __init__(self, coef=0.97):
        super().__init__()
        self.coef = coef
        self.register_buffer("flipped_filter",
                             torch.FloatTensor([-coef, 1.]).unsqueeze(0).unsqueeze(0))

    def forward(self, x):
        x = F.pad(x.unsqueeze(1), (1, 0), "reflect")
        return F.conv1d(x, self.flipped_filter).squeeze(1)

class FbankAug(nn.Module):
    """SpecAugment: one time mask then one frequency mask, per sample."""
    def __init__(self, freq_mask_width=(0, 8), time_mask_width=(0, 10)):
        super().__init__()
        self.freq_mask_width = freq_mask_width
        self.time_mask_width = time_mask_width

    def mask_along_axis(self, x, dim):
        original_size = x.shape
        batch, fea, time = x.shape
        D, width_range = (fea, self.freq_mask_width) if dim == 1 else (time, self.time_mask_width)
        mask_len = torch.randint(width_range[0], width_range[1], (batch, 1),
                                 device=x.device).unsqueeze(2)
        mask_pos = torch.randint(0, max(1, D - mask_len.max()), (batch, 1),
                                 device=x.device).unsqueeze(2)
        arange = torch.arange(D, device=x.device).view(1, 1, -1)
        mask = ((mask_pos <= arange) * (arange < (mask_pos + mask_len))).any(dim=1)
        mask = mask.unsqueeze(2) if dim == 1 else mask.unsqueeze(1)
        return x.masked_fill_(mask, 0.0).view(*original_size)

    def forward(self, x):
        return self.mask_along_axis(self.mask_along_axis(x, dim=2), dim=1)

class ECAPA_TDNN(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.torchfbank = nn.Sequential(
            PreEmphasis(),
            torchaudio.transforms.MelSpectrogram(
                sample_rate=16000, n_fft=512, win_length=400, hop_length=160,
                f_min=20, f_max=7600, window_fn=torch.hamming_window, n_mels=80))
        self.specaug = FbankAug()

        self.conv1  = nn.Conv1d(80, C, kernel_size=5, stride=1, padding=2)
        self.relu   = nn.ReLU()
        self.bn1    = nn.BatchNorm1d(C)
        self.layer1 = Bottle2neck(C, C, kernel_size=3, dilation=2, scale=8)
        self.layer2 = Bottle2neck(C, C, kernel_size=3, dilation=3, scale=8)
        self.layer3 = Bottle2neck(C, C, kernel_size=3, dilation=4, scale=8)
        self.layer4 = nn.Conv1d(3 * C, 1536, kernel_size=1)
        self.attention = nn.Sequential(
            nn.Conv1d(4608, 256, kernel_size=1), nn.ReLU(), nn.BatchNorm1d(256),
            nn.Tanh(), nn.Conv1d(256, 1536, kernel_size=1), nn.Softmax(dim=2))
        self.bn5 = nn.BatchNorm1d(3072)
        self.fc6 = nn.Linear(3072, 192)
        self.bn6 = nn.BatchNorm1d(192)

    def forward(self, x, aug):
        # fbank in fp32 regardless of the surrounding autocast: log() of a fp16
        # mel spectrogram underflows for quiet frames.
        with torch.no_grad(), torch.autocast("cuda", enabled=False):
            x = self.torchfbank(x.float()) + 1e-6
            x = x.log()
            x = x - torch.mean(x, dim=-1, keepdim=True)
            if aug:
                x = self.specaug(x)

        x = self.bn1(self.relu(self.conv1(x)))
        x1 = self.layer1(x)
        x2 = self.layer2(x + x1)
        x3 = self.layer3(x + x1 + x2)
        x  = self.relu(self.layer4(torch.cat((x1, x2, x3), dim=1)))

        t = x.size()[-1]
        global_x = torch.cat((
            x,
            torch.mean(x, dim=2, keepdim=True).repeat(1, 1, t),
            torch.sqrt(torch.var(x, dim=2, keepdim=True).clamp(min=1e-4)).repeat(1, 1, t)),
            dim=1)
        w  = self.attention(global_x)
        mu = torch.sum(x * w, dim=2)
        sg = torch.sqrt((torch.sum((x ** 2) * w, dim=2) - mu ** 2).clamp(min=1e-4))
        return self.bn6(self.fc6(self.bn5(torch.cat((mu, sg), 1))))

_m = ECAPA_TDNN(C)
print(f"ECAPA-TDNN C={C}: "
      f"{sum(p.numel() for p in _m.parameters() if p.requires_grad)/1e6:.2f}M params")
del _m

In [ ]:
# ── AAMSoftmax (same formulation as the reference loss.py and ann_backend.ipynb) ──
class AAMSoftmax(nn.Module):
    def __init__(self, embed_dim, num_classes, m=AAM_M, s=AAM_S):
        super().__init__()
        self.s = s
        self.W = nn.Parameter(torch.empty(num_classes, embed_dim))
        nn.init.xavier_normal_(self.W)
        self.m = m
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb, labels):
        cos = F.linear(F.normalize(emb), F.normalize(self.W)).clamp(-1 + 1e-7, 1 - 1e-7)
        sin = torch.sqrt((1 - cos ** 2).clamp_min(1e-9))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        one_hot = torch.zeros_like(cos).scatter_(1, labels.view(-1, 1), 1.0)
        logits = (one_hot * phi + (1 - one_hot) * cos) * self.s
        return F.cross_entropy(logits, labels)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  EVAL — same protocol as ann_backend.ipynb (session-free all-pairs retrieval)
#  Two changes for the 21.8k-row gallery: vectorized AP (identical math, no
#  per-row GPU sync) and length-bucketed full-utterance embedding extraction.
# ═════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def extract_embeddings(net, bank, cap_sec=TEST_CAP_SEC):
    """Full-utterance embeddings. Length-sorted buckets truncated to the batch
    minimum: no padding, so ASP attention statistics stay exact."""
    net.eval()
    cap = int(cap_sec * SR)
    lens = np.minimum(bank.lengths, cap)
    order = np.argsort(lens)                      # ascending, so trim within a batch is tiny
    out = torch.empty(bank.n, EMBED_DIM, dtype=torch.float32)
    for s in range(0, bank.n, EVAL_BATCH):
        rows = order[s:s + EVAL_BATCH]
        n = int(lens[rows].min())
        n = max(n, CROP_SAMPLES)                  # guard: never shorter than one crop
        batch = np.stack([bank.crop(int(i), n) if lens[i] < n else bank.raw(int(i), cap)[:n]
                          for i in rows])
        x = torch.from_numpy(batch).to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            e = net(x, aug=False)
        out[torch.from_numpy(rows)] = F.normalize(e.float(), dim=1).cpu()
    return out

def _eer_from_hist(gen_hist, imp_hist):
    g  = gen_hist / max(gen_hist.sum(), 1)
    im = imp_hist / max(imp_hist.sum(), 1)
    frr = np.cumsum(g)               # genuine in bins <= thr -> rejected
    far = 1.0 - np.cumsum(im)        # impostor in bins  > thr -> accepted
    k = int(np.argmin(np.abs(frr - far)))
    return float((frr[k] + far[k]) / 2)

@torch.no_grad()
def retrieval_metrics(embs, labels, record_ids, session_free=True, block=512, n_bins=20000):
    N = embs.shape[0]
    E   = embs.to(device)
    lab = torch.as_tensor(labels, device=device)
    rec = torch.as_tensor(np.unique(record_ids, return_inverse=True)[1], device=device)
    gen_hist = np.zeros(n_bins); imp_hist = np.zeros(n_bins)
    r1 = r5 = 0; ap_sum = 0.0; valid = 0
    ranks = torch.arange(1, N + 1, device=device, dtype=torch.float32).view(1, -1)

    for s in range(0, N, block):
        e    = E[s:s + block]
        sims = e @ E.t()
        b    = sims.shape[0]
        rows = torch.arange(s, s + b, device=device)
        if session_free:
            excl = rec[rows][:, None] == rec[None, :]     # same recording (incl. self)
        else:
            excl = rows[:, None] == torch.arange(N, device=device)[None, :]
        sims = sims.masked_fill(excl, -2.0)
        same = (lab[rows][:, None] == lab[None, :]) & (~excl)

        order = torch.argsort(sims, dim=1, descending=True)
        same_sorted = torch.gather(same, 1, order)
        r1 += same_sorted[:, 0].sum().item()
        r5 += same_sorted[:, :5].any(dim=1).sum().item()

        # Vectorized AP: mean over hits of precision@rank. Identical to the
        # per-row loop in ann_backend.ipynb, without the per-row sync.
        rel  = same.sum(dim=1)
        hits = torch.cumsum(same_sorted.float(), dim=1)
        prec = hits / ranks
        ap   = (prec * same_sorted.float()).sum(dim=1) / rel.clamp(min=1)
        ok   = rel > 0
        ap_sum += float(ap[ok].sum()); valid += int(ok.sum())

        gen_hist += torch.histc(sims[same],              bins=n_bins, min=-1, max=1).cpu().numpy()
        imp_hist += torch.histc(sims[(~same) & (~excl)], bins=n_bins, min=-1, max=1).cpu().numpy()

    return dict(rank1=r1 / max(valid, 1), rank5=r5 / max(valid, 1),
                mAP=ap_sum / max(valid, 1), eer=_eer_from_hist(gen_hist, imp_hist))

def evaluate(net, bank, labels):
    embs = extract_embeddings(net, bank)
    return retrieval_metrics(embs, labels, bank.record_ids, session_free=True)

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
#  TRAIN
# ═════════════════════════════════════════════════════════════════════════════
def rss_gb():
    with open("/proc/self/statm") as f:
        return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 1e9

def banner():
    print("===== RUN CONFIG =====")
    print(f"  ECAPA-TDNN C={C}  embed={EMBED_DIM}  batch={BATCH_SIZE}  epochs={EPOCHS}")
    print(f"  Adam lr={LR} wd={WEIGHT_DECAY} decay={LR_DECAY}/epoch | AAM m={AAM_M} s={AAM_S}")
    print(f"  crop={CROP_SEC}s  specaug=on  musan/rir={'on' if wave_aug.active else 'off'}")
    print(f"  train {train_bank.n} clips / {NUM_CLASSES} spk | eval {test_bank.n} clips")
    print("======================")

def train():
    net = ECAPA_TDNN(C).to(device)
    aam = AAMSoftmax(EMBED_DIM, NUM_CLASSES).to(device)
    opt = torch.optim.Adam(list(net.parameters()) + list(aam.parameters()),
                           lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=LR_DECAY)
    scaler = torch.amp.GradScaler("cuda")

    history, best_eer, start_epoch = [], float("inf"), 0
    banner()

    if RESUME and os.path.exists(LAST_CKPT):
        ck = torch.load(LAST_CKPT, map_location=device)
        if ck.get("tag") == TAG:
            net.load_state_dict(ck["model"]); aam.load_state_dict(ck["aam"])
            opt.load_state_dict(ck["opt"]);   sched.load_state_dict(ck["sched"])
            scaler.load_state_dict(ck["scaler"])
            history, best_eer, start_epoch = ck["history"], ck["best_eer"], ck["epoch"] + 1
            print(f"[resume] epoch {start_epoch}  best_eer={best_eer:.4f}")
        else:
            print(f"[resume] tag mismatch ({ck.get('tag')} != {TAG}); fresh start")

    y_t = torch.from_numpy(y_train)
    steps = train_bank.n // BATCH_SIZE
    rng = np.random.default_rng(SEED + start_epoch)

    for epoch in range(start_epoch, EPOCHS):
        net.train()
        perm = rng.permutation(train_bank.n)
        t0, running, nb = time.time(), 0.0, 0

        for si in range(steps):
            idx = perm[si * BATCH_SIZE:(si + 1) * BATCH_SIZE]
            x = train_bank.batch(idx).to(device, non_blocking=True)
            y = y_t[idx].to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                emb = net(x, aug=True)
            loss = aam(emb.float(), y)                      # AAM in fp32
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(
                list(net.parameters()) + list(aam.parameters()), GRAD_CLIP)
            scaler.step(opt); scaler.update()
            running += loss.item(); nb += 1

            if (si + 1) % 200 == 0:
                print(f"  [e{epoch:02d} {si+1}/{steps}] loss={running/nb:.4f} "
                      f"({time.time()-t0:.0f}s)", flush=True)
        sched.step()

        rec = dict(epoch=epoch, loss=running / max(nb, 1),
                   lr=opt.param_groups[0]["lr"], rss=rss_gb())
        improved = False
        if (epoch + 1) % EVAL_EVERY == 0 or epoch == EPOCHS - 1:
            m = evaluate(net, test_bank, y_test)
            rec["metrics"] = m
            improved = m["eer"] < best_eer
            if improved:
                best_eer = m["eer"]
                torch.save(dict(tag=TAG, epoch=epoch, metrics=m,
                                model=net.state_dict(), aam=aam.state_dict()), BEST_CKPT)
            print(f"[e{epoch:02d}] loss={rec['loss']:.4f} lr={rec['lr']:.2e} | "
                  f"eer={m['eer']:.4f} r1={m['rank1']:.3f} r5={m['rank5']:.3f} "
                  f"mAP={m['mAP']:.3f} | RSS={rec['rss']:.1f}GB "
                  f"{'*best' if improved else ''} ({time.time()-t0:.0f}s)")
        else:
            print(f"[e{epoch:02d}] loss={rec['loss']:.4f} lr={rec['lr']:.2e} | "
                  f"RSS={rec['rss']:.1f}GB ({time.time()-t0:.0f}s)")

        history.append(rec)
        torch.save(dict(tag=TAG, epoch=epoch, best_eer=best_eer, history=history,
                        model=net.state_dict(), aam=aam.state_dict(),
                        opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict()), LAST_CKPT)

    print(f"[done] best EER={best_eer:.4f}  ->  {BEST_CKPT}")
    return history

In [ ]:
# ===================== RUN =====================
history = train()